[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). In this notebook, we'll use [ChatOllama](https://docs.langchain.com/oss/python/integrations/chat/ollama) so you can run models locally without an `OPENAI_API_KEY`.

Before continuing, make sure Ollama is running locally and that you have pulled a model such as `llama3.1`.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain-ollama langchain_core langchain_community langchain-tavily

In [2]:
import os
import urllib.error
import urllib.request
import json

os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")

def _get_installed_ollama_models():
    host = os.environ["OLLAMA_HOST"].rstrip("/")
    try:
        with urllib.request.urlopen(f"{host}/api/tags") as response:
            payload = json.load(response)
    except urllib.error.URLError as exc:
        raise RuntimeError(
            f"Could not reach Ollama at {host}. Start Ollama first, then rerun this cell."
        ) from exc

    models = [model["name"] for model in payload.get("models", [])]
    if not models:
        raise RuntimeError(
            "No Ollama models are installed. Run `ollama pull llama3.1` or set OLLAMA_MODEL to an installed model."
        )
    return models

installed_models = _get_installed_ollama_models()
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL") or installed_models[0]
OLLAMA_MODEL_ALT = os.environ.get("OLLAMA_MODEL_ALT") or installed_models[min(1, len(installed_models) - 1)]

print(f"Using Ollama at {os.environ['OLLAMA_HOST']}")
print(f"Installed models: {installed_models}")
print(f"Primary model: {OLLAMA_MODEL}")
print(f"Secondary model: {OLLAMA_MODEL_ALT}")

Using Ollama at http://localhost:11434
Installed models: ['llama3.1:8b']
Primary model: llama3.1:8b
Secondary model: llama3.1:8b


[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-ollama` package. With this, we can instantiate a `ChatOllama` model object backed by your local Ollama server. The notebook will use `OLLAMA_MODEL` / `OLLAMA_MODEL_ALT` if set, otherwise it will detect your installed Ollama models automatically and pick from them.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [3]:
from langchain_ollama import ChatOllama

primary_chat = ChatOllama(model=OLLAMA_MODEL, temperature=0)
secondary_chat = ChatOllama(model=OLLAMA_MODEL_ALT, temperature=0)

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [4]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
primary_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-16T13:37:39.335098Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3291768000, 'load_duration': 2854740667, 'prompt_eval_count': 12, 'prompt_eval_duration': 232491167, 'eval_count': 10, 'eval_duration': 196777541, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cf6dd-c928-7b42-8ce5-bb23924bc43d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 10, 'total_tokens': 22})

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [5]:
primary_chat.invoke("hello world")

AIMessage(content='Hello World! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-16T13:40:29.517196Z', 'done': True, 'done_reason': 'stop', 'total_duration': 624773417, 'load_duration': 76045084, 'prompt_eval_count': 12, 'prompt_eval_duration': 326744708, 'eval_count': 11, 'eval_duration': 217561290, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cf6e0-6c5a-74a1-8481-6fbc17d47d6b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 11, 'total_tokens': 23})

In [6]:
secondary_chat.invoke("hello world")

AIMessage(content='Hello World! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-03-16T13:40:32.726587Z', 'done': True, 'done_reason': 'stop', 'total_duration': 455388750, 'load_duration': 77091333, 'prompt_eval_count': 12, 'prompt_eval_duration': 157080208, 'eval_count': 11, 'eval_duration': 216052583, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--019cf6e0-798d-7b70-93b4-f616c62215d6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 11, 'total_tokens': 23})

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [ ]:
import getpass

def _set_env(var: str) -> None:
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"Enter value for {var}: ")

_set_env("TAVILY_API_KEY")

In [11]:
from langchain_tavily import TavilySearch  # updated at 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [12]:
search_docs

[{'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It?',
  'content': 'LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. By managing the flow of data and the sequence of operations, LangGraph allows developers to focus on the high-level logic of their applications rather than the intricacies of agent coordination. Whether you need a chatbot that can handle various types of user requests or a multi-agent system that performs complex tasks, LangGraph provides the tools to build exactly what you need. LangGraph significantly simplifies the development of complex LLM applications by providing a structured framework for managing state and coordinating agent interactions.',
  'score': 0.9581988,
  'raw_content': None},
 {'url': 'https://www.geeksforgeeks.org/machine-learning